In [ ]:
# hack to change jupyter directory in notebooks for imports and handle asyncio
import os
from pathlib import Path
import pandas as pd
from ib_insync import util
if Path.cwd().parts[-1] == 'notebooks':
  root = Path.cwd().parent
  import nest_asyncio
  nest_asyncio.apply()
  util.startLoop()
  pd.options.display.max_columns = None
  pd.options.display.float_format = '{:,.2f}'.format # set float precision with comma
else:
  root = Path.cwd()
os.chdir(root)

# Logger
import logging
log = logging.getLogger('ib_log')

In [ ]:
datapath = root / 'data' / 'master'

In [ ]:
# Why are commissions incorrect?
from src.support import qualifyAsync, marginsAsync, get_dte, get_closest, get_chain_margins, nse2ib_symbol_convert
from src.nse import get_nse_syms, nse2ib_symbol_convert, get_prices
# from ib_insync import IB, Contract, MarketOrder
# import asyncio
# import numpy as np

# import nest_asyncio
# nest_asyncio.apply()

# ib = IB()
port = 3000

df_syms = get_nse_syms()
df_chains = pd.read_pickle(datapath / 'nse_chains.pkl')
df_unds = pd.read_pickle(datapath / 'nse_margins.pkl')
df_opt_margin = pd.read_pickle(datapath / 'nse_opt_margins.pkl')


In [ ]:
# prepare columns
cols = ['conId', 'symbol', 'expiry', 'strike', 'right', 'undPrice', 'dte', 
        'opt_iv', 'lastPrice', 'bid', 'ask', 'lot', 'margin', 'comm']
df = df_opt_margin[cols]

# get options with ivs
df = df[df.opt_iv>0]

# Get relevant calls and puts
cs = (df.strike > df.lastPrice) & (df.right == 'C')
ps = (df.strike < df.lastPrice) & (df.right == 'P')
df = df[cs | ps]

# Get the standard multiples
import math
sdevmult = abs((df.strike - df.undPrice) / (
  (df.dte / 365).apply(math.sqrt) * df.opt_iv * df.undPrice))
  
# set the standard deviations
m = sdevmult > 3
df = df.loc[m]

In [ ]:
asc = ((df.lastPrice*df.lot-df.comm)/df.margin*365/df.dte).sort_values(ascending=False)
df.loc[asc.index].assign(rom=asc)

In [ ]:
# Get history
df_hist = pd.read_pickle(datapath / 'nse_hists.pkl')

In [ ]:
g = df_hist.groupby('symbol')

In [ ]:
import math
g.close.pct_change().expanding(1).std(ddof=0)*math.sqrt(252)

In [ ]:
import numpy as np
g.close.pct_change().apply(lambda x: np.log(1+x)).std()